# 04 Model 2: RNN + Classifier
Training of architecture 2: the same frozen word embeddings as in model 1, a 2-layer RNN (LSTM or GRU) and a 2-layer classifier (ReLU non-linearity). The RNN and the classifier are trained. Experiments are tracked with Weights & Biases.

**In short.** Question: does an RNN encoder beat averaging with the same fixed-position input, and which settings work best? Answer: on the validation set yes, in this format; `04b_preprocessing_experiments.ipynb` shows that the input format also matters. Chosen: GRU, RNN hidden size 128, learning rate 0.001, dropout 0.5, batch size 32, best epoch 4 (validation loss 0.5872, accuracy 67.60%).

Load the token indices, the labels and the embedding matrix that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only evaluated once at the end, in `05_evaluation.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model2.npz")
# token indices of O1, O2, h1, h2 for every example, and the labels (0 = h1, 1 = h2)
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
# GloVe vectors: row i = vector of the word with index i (row 0 = padding)
embedding_matrix = torch.tensor(data["embedding_matrix"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)
print(embedding_matrix.shape, embedding_matrix.dtype)

torch.Size([16801, 4, 34]) torch.int64
torch.Size([16801]) torch.int64
torch.Size([15465, 300]) torch.float32


## Classifier
The same 2-layer classifier as for model 1 (`03_model1_embeddings.ipynb`, from the course exercise "PyTorch" (Week 1)): linear layer, ReLU, linear layer. In model 2, its input are the final hidden states of the RNN instead of the averaged word vectors.

In [2]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

## Model 2: RNN + classifier
The same LSTM (or GRU) reads each of the four texts (O1, O2, h1, h2); the final hidden state of its last layer, h_n, is the representation of the text (course slides "Advanced RNNs", slide 16). The four final hidden states are put side by side, like the four sentence vectors in model 1, and the classifier makes the decision. The embedding layer contains the frozen GloVe vectors (`freeze=True`), and index 0 is the padding (`padding_idx=0`). Each text is passed to the RNN with its real length, the number of token indices that are not 0, using `pack_padded_sequence` (course slides "Project Discussions" (Week 4), slide 17), so the RNN stops after the last real word and h_n is the state after that word (see the padding section in `02_preprocessing.ipynb`). Dropout with the probability `dropout` can be added between the two RNN layers and on the final hidden states before the classifier (see the section "Dropout"); by default it is 0.

In [3]:
from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # dropout on the 4 final hidden states before the classifier (0.0 = no dropout)
        self.dropout = nn.Dropout(dropout)
        # classifier on the 4 final hidden states side by side (O1, O2, h1, h2)
        self.classifier = Classifier(4 * hidden_dim, 2, inner_dim=inner_dim)

    def forward(self, x):
        # x: token indices, shape [batch size, 4, 34] (O1, O2, h1, h2)
        features = []
        for i in range(4):
            # one text (O1, O2, h1 or h2) of every example in the batch: [batch size, 34]
            text = x[:, i]
            # real length of each text = number of token indices that are not 0 (padding)
            lengths = (text != 0).sum(dim=1)
            # look up the GloVe vectors: [batch size, 34, 300]
            embedded = self.embedding(text)
            # give the RNN the real lengths, so it stops after the last real word
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            # the RNN reads the text; only the final hidden state is needed
            _, hidden = self.rnn(packed)
            # the LSTM returns (h_n, c_n), the GRU only h_n
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            # h_n of the last layer: [batch size, hidden dim]
            features.append(hidden[-1])
        # the 4 final hidden states side by side: [batch size, 4 * hidden dim]
        features = torch.concat(features, dim=-1)
        # dropout on the final hidden states (active only in training mode)
        features = self.dropout(features)
        return self.classifier(features)

# same start weights every time (as in model 1)
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
print(model)

RNNClassifier(
  (embedding): Embedding(15465, 300, padding_idx=0)
  (rnn): LSTM(300, 128, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (classifier): Classifier(
    (linear1): Linear(in_features=512, out_features=100, bias=True)
    (relu): ReLU()
    (linear2): Linear(in_features=100, out_features=2, bias=True)
  )
)


Test the untrained model on the first 5 training examples: one pair of logits (h1, h2) per example.

In [4]:
# 5 examples, each with 4 texts of 34 token indices
x = X_train[:5]
# forward pass: one pair of logits per example
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 4, 34])
torch.Size([5, 2])
tensor([[0.0967, 0.0726],
        [0.0921, 0.0704],
        [0.0973, 0.0738],
        [0.0959, 0.0718],
        [0.0912, 0.0662]], grad_fn=<AddmmBackward0>)


## Loss, optimizer and batches
The same loss, optimizer and batches as for model 1 (`03_model1_embeddings.ipynb`), so that the difference between the two models comes from the RNN.

In [5]:
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# cross-entropy loss on the 2 logits (applies the softmax itself)
criterion = nn.CrossEntropyLoss()

# training batches: 32 examples, shuffled in every epoch, the last batch (1 example) is skipped
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
# equal batches for measuring loss and accuracy (16801 = 53 x 317, 1000 = 10 x 100), not shuffled
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

# number of batches, and the shape of one training batch: 32 examples, 4 texts, 34 token indices
x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 4, 34]) torch.Size([32])


## Evaluation
The same `evaluate()` as for model 1. Check: an untrained model should reach about 0.69 loss and 50% accuracy. Before training, its logit for h1 is slightly higher than the one for h2 for every example, so it predicts h1 for all examples, and the accuracy is the share of h1 in the validation set: 508 of 1000, 50.80% (the same value as the collapsed model 1 run with learning rate 0.01).

In [6]:
# same function as in 03_model1_embeddings.ipynb
def evaluate(model, loader):
    # evaluation mode, and no gradients are needed
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            # loss of this batch
            losses.append(criterion(output, y).item())
            # predicted class: the hypothesis with the higher logit
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    # join the batches: one prediction and one label per example
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    # share of correct predictions
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

# untrained model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6931, accuracy: 50.80%


## Check: overfit one batch
Before the long training runs, check that the model can learn at all: a fresh model is trained on one fixed batch of 32 training examples for 50 steps. The loss on this batch must go down to almost 0 and the accuracy up to 100%; otherwise there is a bug, for example no gradient through the packed sequences ("overfit one batch", Karpathy, 2019). At the end, check that the embedding layer still contains exactly the GloVe vectors, i.e. that the embeddings are really frozen.

In [7]:
# fresh model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
optimizer = Adam(model.parameters(), lr=0.001)
# one fixed batch: the first 32 training examples
x, y = X_train[:32], y_train[:32]

model.train()
for step in range(1, 51):
    # one training step, as for model 1
    output = model(x)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # loss and accuracy on the batch every 10 steps
    if step % 10 == 0:
        accuracy = (output.argmax(dim=1) == y).float().mean().item()
        print(f"Step {step}: loss {loss.item():.4f}, accuracy {accuracy:.2%}")

# frozen embeddings: the embedding layer must still be exactly the GloVe matrix
print("Embeddings unchanged:", torch.equal(model.embedding.weight, embedding_matrix))

Step 10: loss 0.6012, accuracy 56.25%
Step 20: loss 0.0390, accuracy 100.00%
Step 30: loss 0.0001, accuracy 100.00%
Step 40: loss 0.0000, accuracy 100.00%
Step 50: loss 0.0000, accuracy 100.00%
Embeddings unchanged: True


The loss on the batch goes down to 0.0000 and the accuracy reaches 100% after 20 steps, and the embedding layer is unchanged: the model can learn, and the GloVe vectors stay frozen.

## Experiment tracking
The training runs are tracked with W&B, as for model 1 (same project `nlp-project1-art-final`, and no W&B log lines in the notebook output).

In [8]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login()


True

## Training
The same training function as for model 1 (`03_model1_embeddings.ipynb`). The hyperparameters are the type of RNN (`lstm` or `gru`), its hidden size and the learning rate.

- **Early stopping** (course slides "Project Discussions" (Week 6), slide 38): the training stops when the validation loss has not improved for 3 epochs in a row, after at most 15 epochs. One epoch takes about 1 to 3 minutes on the CPU (model 1: a few seconds), so the number of epochs is limited.
- **Patience 3:** the validation loss is noisy. In the model 1 run with 100 hidden units and learning rate 0.001, it was 0.6494, 0.6572 and 0.6500 in epochs 8 to 10, so a single worse epoch should not stop the training. `04c_patience_check.ipynb` repeats the chosen configuration with patience 10: its best epoch stays epoch 4.
- **Regularization:** no dropout in the first experiments, as in model 1, so that both models are trained the same way; tested in the section "Dropout". No weight decay, as in model 1 (same slides, slide 34); it is not tested, to limit the number of runs (same slides (Week 7), slide 55).
- **Normalization:** none in the chosen configuration (same slides, slide 34). Layer normalization is tested once, to limit the number of runs (same slides (Week 7), slide 55), and only in the classifier, because `nn.GRU` has no option for it.

In [9]:
import os
import time

os.makedirs("checkpoints", exist_ok=True)

def train(rnn_type="lstm", hidden_dim=128, learning_rate=0.001, dropout=0.0, max_epochs=15, patience=3):
    # run name, e.g. glove-lstm-h128-lr0.001, with dropout e.g. glove-lstm-h128-lr0.001-d0.2
    name = f"glove-{rnn_type}-h{hidden_dim}-lr{learning_rate}"
    if dropout > 0:
        name += f"-d{dropout}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art-final",
        name=name,
        config={
            "model": f"glove-{rnn_type}",
            "rnn_type": rnn_type,
            "hidden_dim": hidden_dim,
            "num_layers": 2,
            "inner_dim": 100,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "max_epochs": max_epochs,
            "patience": patience,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    # fresh model with the same start weights in every run
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, dropout=dropout)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        start = time.time()
        # one training step per batch, as for model 1
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # loss and accuracy after this epoch
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        minutes = (time.time() - start) / 60
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                print(f"Early stopping: no lower validation loss in the last {patience} epochs")
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    # return the model of the best epoch, not the one of the last epoch
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

## Experiments
Hyperparameter tuning on the validation set, as for model 1. The experiments answer three questions:
1. **Does an RNN encoder beat averaging?** Model 1 averages the word vectors of each text (chosen run: validation loss 0.6445, accuracy 63.20%).
2. **LSTM or GRU?** Both have a similar performance, and which one is better depends on the task and the dataset (course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68).
3. **Is the learning rate 0.01 too high?** The learning rate is "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32). For model 1, 0.01 made most ReLU units inactive.

This grid search (all combinations, course slides "Project Discussions" (Week 7), slide 44) gives 4 runs: LSTM and GRU, each with learning rate 0.001 and 0.01; hidden size 128, 2 layers, no dropout, batch size 32, seed 42. All combinations are trained, because the best learning rate can be different for the LSTM and the GRU. Question 1 needs no extra run: the best of the 4 runs is compared with model 1. The RNN hidden size (128) and the 100 hidden units of the classifier are kept fixed to limit the number of runs, because one epoch takes about 1 to 3 minutes on the CPU. The number of hidden units of the classifier changed the validation scores of model 1 little (all runs that learned: 60.20% to 63.30% validation accuracy). Each configuration is trained once (one seed), as for model 1, so small differences can be chance.

In [10]:
# 4 runs: LSTM and GRU, each with learning rate 0.001 and 0.01
for rnn_type in ["lstm", "gru"]:
    for learning_rate in [0.001, 0.01]:
        train(rnn_type=rnn_type, learning_rate=learning_rate)

Epoch 1: train loss 0.6658, train acc 66.06%, valid loss 0.6873, valid acc 62.60% (1.8 min)
Epoch 2: train loss 0.5446, train acc 71.74%, valid loss 0.6050, valid acc 66.70% (2.0 min)
Epoch 3: train loss 0.4742, train acc 77.44%, valid loss 0.6064, valid acc 66.80% (2.6 min)
Epoch 4: train loss 0.3904, train acc 83.83%, valid loss 0.6116, valid acc 67.60% (3.4 min)
Epoch 5: train loss 0.2898, train acc 89.23%, valid loss 0.6496, valid acc 67.80% (2.7 min)
Early stopping: no lower validation loss in the last 3 epochs
Best epoch: 2, valid loss 0.6050, valid acc 66.70%
Epoch 1: train loss 0.6931, train acc 50.36%, valid loss 0.6931, valid acc 50.80% (2.6 min)
Epoch 2: train loss 0.6937, train acc 49.64%, valid loss 0.6939, valid acc 49.20% (2.4 min)
Epoch 3: train loss 0.6932, train acc 50.36%, valid loss 0.6930, valid acc 50.80% (2.1 min)
Epoch 4: train loss 0.6934, train acc 49.64%, valid loss 0.6935, valid acc 49.20% (2.0 min)
Epoch 5: train loss 0.6933, train acc 49.64%, valid loss 0.

## Dropout
All runs with learning rate 0.001 overfit quickly: the training accuracy reaches 89-95% after 5-6 epochs, while the validation loss rises again after epoch 2 or 3. Question: **does dropout reduce the overfitting and lower the validation loss?**

Dropout (Srivastava et al., 2014; course slides "Preliminaries" (Week 1), slide 20; "Project Discussions" (Week 6), slide 34) is used in two places, not on recurrent connections, as Zaremba et al. (2014) recommend for LSTMs: between the two RNN layers (argument `dropout` of `nn.LSTM` and `nn.GRU`) and on the four final hidden states before the classifier (`nn.Dropout`). There is no dropout after the output, because then the model has no way to correct it ("Project Discussions", slide 34). With dropout 0, `nn.Dropout` changes nothing, so the runs above remain valid.

4 runs: LSTM and GRU with learning rate 0.001 (0.01 did not learn), each with dropout 0.2 and 0.5; everything else as above. They are compared with the runs without dropout.

In [11]:
# 4 runs: LSTM and GRU with learning rate 0.001, each with dropout 0.2 and 0.5
for rnn_type in ["lstm", "gru"]:
    for dropout in [0.2, 0.5]:
        train(rnn_type=rnn_type, learning_rate=0.001, dropout=dropout)

Epoch 1: train loss 0.5993, train acc 66.73%, valid loss 0.6312, valid acc 64.00% (3.3 min)
Epoch 2: train loss 0.5539, train acc 71.09%, valid loss 0.6105, valid acc 68.00% (3.1 min)
Epoch 3: train loss 0.4996, train acc 75.70%, valid loss 0.6189, valid acc 66.50% (3.5 min)
Epoch 4: train loss 0.4009, train acc 81.96%, valid loss 0.6353, valid acc 67.30% (3.4 min)
Epoch 5: train loss 0.3327, train acc 87.40%, valid loss 0.6492, valid acc 66.90% (3.3 min)
Early stopping: no lower validation loss in the last 3 epochs
Best epoch: 2, valid loss 0.6105, valid acc 68.00%
Epoch 1: train loss 0.6079, train acc 66.23%, valid loss 0.6344, valid acc 61.50% (3.2 min)
Epoch 2: train loss 0.5619, train acc 70.60%, valid loss 0.6078, valid acc 66.10% (3.3 min)
Epoch 3: train loss 0.5181, train acc 73.92%, valid loss 0.5973, valid acc 65.60% (3.0 min)
Epoch 4: train loss 0.4517, train acc 78.85%, valid loss 0.6088, valid acc 65.50% (2.9 min)
Epoch 5: train loss 0.3929, train acc 83.63%, valid loss 0.

Dropout 0.5, the largest value tested, gives the lowest validation loss for both RNNs, so it is on the boundary of the tested range. Following slide 53 of the course slides "Project Discussions" (Week 7), the next larger value, 0.7, is tested too.

In [12]:
# the next larger dropout, because 0.5 (the largest value tested) was the best
for rnn_type in ["lstm", "gru"]:
    train(rnn_type=rnn_type, learning_rate=0.001, dropout=0.7)

Epoch 1: train loss 0.6423, train acc 63.07%, valid loss 0.6520, valid acc 60.90% (2.0 min)
Epoch 2: train loss 0.5940, train acc 66.84%, valid loss 0.6178, valid acc 64.70% (2.1 min)
Epoch 3: train loss 0.5728, train acc 70.90%, valid loss 0.6165, valid acc 65.40% (2.1 min)
Epoch 4: train loss 0.5251, train acc 74.00%, valid loss 0.6005, valid acc 66.10% (2.2 min)
Epoch 5: train loss 0.4834, train acc 78.07%, valid loss 0.5950, valid acc 66.60% (2.2 min)
Epoch 6: train loss 0.4200, train acc 80.30%, valid loss 0.6092, valid acc 68.10% (2.3 min)
Epoch 7: train loss 0.3580, train acc 84.73%, valid loss 0.6337, valid acc 66.80% (2.3 min)
Epoch 8: train loss 0.3194, train acc 87.28%, valid loss 0.6678, valid acc 65.30% (2.0 min)
Early stopping: no lower validation loss in the last 3 epochs
Best epoch: 5, valid loss 0.5950, valid acc 66.60%
Epoch 1: train loss 0.6401, train acc 62.65%, valid loss 0.6538, valid acc 59.30% (2.0 min)
Epoch 2: train loss 0.5961, train acc 66.97%, valid loss 0.

## Learning rate 0.0001
As for model 1, the best learning rate of the grid, 0.001, is its smallest value, on the boundary of the tested range, so the next smaller value in exponential steps, 0.0001, is tested too (course slides "Project Discussions" (Week 7), slides 52 and 53), for the LSTM and the GRU without dropout. With 10 times smaller update steps, more epochs may be needed, so the maximum is 30 epochs instead of 15 (slide 41); early stopping ends the runs as before. The learning rate and the dropout are tuned one after the other, not in all combinations, which may miss the best combination (slide 41).

In [13]:
# the next smaller learning rate, without dropout, with at most 30 epochs (smaller steps)
for rnn_type in ["lstm", "gru"]:
    train(rnn_type=rnn_type, learning_rate=0.0001, max_epochs=30)

Epoch 1: train loss 0.6509, train acc 62.38%, valid loss 0.6584, valid acc 60.70% (2.2 min)
Epoch 2: train loss 0.6145, train acc 66.48%, valid loss 0.6381, valid acc 64.20% (2.2 min)
Epoch 3: train loss 0.5932, train acc 67.85%, valid loss 0.6299, valid acc 64.00% (2.2 min)
Epoch 4: train loss 0.5765, train acc 69.51%, valid loss 0.6235, valid acc 64.70% (2.1 min)
Epoch 5: train loss 0.5597, train acc 70.79%, valid loss 0.6264, valid acc 64.30% (2.1 min)
Epoch 6: train loss 0.5445, train acc 71.89%, valid loss 0.6277, valid acc 65.40% (2.2 min)
Epoch 7: train loss 0.5250, train acc 73.56%, valid loss 0.6360, valid acc 64.50% (2.1 min)
Early stopping: no lower validation loss in the last 3 epochs
Best epoch: 4, valid loss 0.6235, valid acc 64.70%
Epoch 1: train loss 0.6512, train acc 61.69%, valid loss 0.6580, valid acc 61.60% (2.0 min)
Epoch 2: train loss 0.6091, train acc 66.36%, valid loss 0.6310, valid acc 64.70% (2.0 min)
Epoch 3: train loss 0.5945, train acc 67.72%, valid loss 0.

## Layer normalization
Does layer normalization lower the validation loss of the chosen configuration of model 2? As for model 1 (`03_model1_embeddings.ipynb`, section "Layer normalization"), the chosen configuration (GRU, learning rate 0.001, dropout 0.5, at most 15 epochs, patience 3) is trained once more with `nn.LayerNorm` between the first linear layer of the classifier and the ReLU. `nn.GRU` has no option for layer normalization, so this tests it only in the classifier, not inside the RNN as in the layer normalized RNN of Ba et al. (2016). The check below shows that both runs start from the same weights and get the same first batch. As for all runs, the best epoch is selected by the validation loss, and if this run has the lowest validation loss of all candidate runs, it becomes the chosen configuration. The training steps are the same as in `train()`; only the class differs.

In [14]:
# model 2 with layer norm in the classifier: everything else stays the same
class RNNLayerNormClassifier(RNNClassifier):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        # build the normal model first, so the random starting weights stay the same
        super().__init__(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout, inner_dim=inner_dim)
        # new: linear1 is now followed by layer norm, before the ReLU of the classifier
        self.classifier.linear1 = nn.Sequential(self.classifier.linear1, nn.LayerNorm(inner_dim))


In [15]:
# same seed as in train(): the model with layer norm, then the first batch
torch.manual_seed(42)
model_ln = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", dropout=0.5)
x_ln, _ = next(iter(train_loader))

# the same for the model without layer norm
torch.manual_seed(42)
model_plain = RNNClassifier(embedding_matrix, rnn_type="gru", dropout=0.5)
x_plain, _ = next(iter(train_loader))

# same starting weights in the RNN and the classifier, and same first batch?
print(torch.equal(model_ln.rnn.weight_ih_l0, model_plain.rnn.weight_ih_l0))
print(torch.equal(model_ln.classifier.linear1[0].weight, model_plain.classifier.linear1.weight))
print(torch.equal(model_ln.classifier.linear2.weight, model_plain.classifier.linear2.weight))
print(torch.equal(x_ln, x_plain))
# how many parameters does layer norm add?
print(sum(p.numel() for p in model_ln.parameters()) - sum(p.numel() for p in model_plain.parameters()))


True
True
True
True
200


In [16]:
import time

# the chosen configuration of model 2, with layer norm in the classifier
rnn_type, hidden_dim, learning_rate, dropout, max_epochs, patience = "gru", 128, 0.001, 0.5, 15, 3
name = f"glove-{rnn_type}-ln-h{hidden_dim}-lr{learning_rate}-d{dropout}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-final",
    name=name,
    config={
        "model": f"glove-{rnn_type}-ln",
        "rnn_type": rnn_type,
        "hidden_dim": hidden_dim,
        "num_layers": 2,
        "inner_dim": 100,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "max_epochs": max_epochs,
        "patience": patience,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with the layer norm class
torch.manual_seed(42)
model = RNNLayerNormClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim, dropout=dropout)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(max_epochs):
    start = time.time()
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    minutes = (time.time() - start) / 60
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, without layer norm (chosen run): best epoch 4, valid loss 0.5872, valid acc 67.60%")

Epoch 1: train loss 0.6462, train acc 63.54%, valid loss 0.6590, valid acc 60.70% (2.0 min)
Epoch 2: train loss 0.5913, train acc 67.94%, valid loss 0.6171, valid acc 65.10% (2.0 min)
Epoch 3: train loss 0.5529, train acc 71.70%, valid loss 0.6051, valid acc 66.40% (2.2 min)
Epoch 4: train loss 0.5159, train acc 75.79%, valid loss 0.5931, valid acc 65.30% (2.2 min)
Epoch 5: train loss 0.4450, train acc 80.17%, valid loss 0.6171, valid acc 66.30% (2.2 min)
Epoch 6: train loss 0.3818, train acc 83.80%, valid loss 0.6154, valid acc 66.80% (2.0 min)
Epoch 7: train loss 0.3168, train acc 87.90%, valid loss 0.6589, valid acc 64.40% (2.1 min)
Early stopping: no lower validation loss in the last 3 epochs
Best epoch: 4, valid loss 0.5931, valid acc 65.30%
For comparison, without layer norm (chosen run): best epoch 4, valid loss 0.5872, valid acc 67.60%


## Results
Load the best checkpoint of every model 2 run, evaluate it on the validation set, and sort the results by the validation loss (lowest first), as for model 1.

In [17]:
# all model 2 runs: RNN type, learning rate, dropout
runs = [(rnn_type, learning_rate, dropout) for rnn_type in ["lstm", "gru"]
        for learning_rate, dropout in [(0.0001, 0.0), (0.001, 0.0), (0.01, 0.0), (0.001, 0.2), (0.001, 0.5), (0.001, 0.7)]]

results = []
for rnn_type, learning_rate, dropout in runs:
    # same run name as in train()
    name = f"glove-{rnn_type}-h128-lr{learning_rate}"
    if dropout > 0:
        name += f"-d{dropout}"
    # build the model with the settings of the run and load the weights of its best epoch
    model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, dropout=dropout)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    valid_loss, valid_acc = evaluate(model, valid_loader)
    results.append((valid_loss, valid_acc, name))

# the run with layer norm: the same evaluation, with its own class
name = "glove-gru-ln-h128-lr0.001-d0.5"
model = RNNLayerNormClassifier(embedding_matrix, rnn_type="gru", dropout=0.5)
model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
valid_loss, valid_acc = evaluate(model, valid_loader)
results.append((valid_loss, valid_acc, name))

# lowest validation loss first
for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

glove-gru-h128-lr0.001-d0.5: valid loss 0.5872, valid acc 67.60%
glove-gru-ln-h128-lr0.001-d0.5: valid loss 0.5931, valid acc 65.30%
glove-lstm-h128-lr0.001-d0.7: valid loss 0.5950, valid acc 66.60%
glove-lstm-h128-lr0.001-d0.5: valid loss 0.5973, valid acc 65.60%
glove-gru-h128-lr0.001-d0.7: valid loss 0.5981, valid acc 67.20%
glove-gru-h128-lr0.001: valid loss 0.6001, valid acc 67.40%
glove-lstm-h128-lr0.001: valid loss 0.6050, valid acc 66.70%
glove-gru-h128-lr0.001-d0.2: valid loss 0.6081, valid acc 66.10%
glove-lstm-h128-lr0.001-d0.2: valid loss 0.6105, valid acc 68.00%
glove-gru-h128-lr0.0001: valid loss 0.6193, valid acc 65.90%
glove-lstm-h128-lr0.0001: valid loss 0.6235, valid acc 64.70%
glove-lstm-h128-lr0.01: valid loss 0.6930, valid acc 50.80%
glove-gru-h128-lr0.01: valid loss 0.6930, valid acc 50.80%


## Sweep
The experiments above tuned the learning rate and the dropout one after the other and kept the RNN hidden size fixed, which may miss the best setting (course slides "Project Discussions" (Week 7), slide 41). Question: **does another combination of RNN hidden size, learning rate and dropout lower the validation loss?** A W&B sweep (same slides, slide 57) trains all combinations automatically: RNN hidden size 64, 128 and 256, learning rate 1e-3 and 3.16e-4 (the finer step 1e-3.5, slide 52), and dropout 0.5 and 0.7, 12 runs. Only the GRU is used, and the classifier stays at 100 hidden units, to limit the number of runs (slide 55): with the same settings, the LSTM and the GRU differ by about 0.01 at most (see the section "Results").

The method is `grid`, as for model 1 (`03_model1_embeddings.ipynb`), so that starting the sweep again tests the same configurations. Every run has seed 42, patience 3 and at most 30 epochs, as in the section "Learning rate 0.0001". Two combinations (RNN hidden size 128, learning rate 1e-3, dropout 0.5 and 0.7) repeat earlier runs: their best epoch and validation loss must be the same, which checks that the sweep trains the same way as `train()`. The selection rule stays the same: a sweep run with the lowest validation loss of all candidate runs becomes the chosen configuration. Selecting among more runs also makes that validation loss more optimistic.

In [18]:
# the sweep: all combinations of these values (3 x 2 x 2 = 12 runs)
sweep_config = {
    "name": "model2-grid",
    "method": "grid",
    # the value in the run summary by which W&B compares the runs
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "hidden_dim": {"values": [64, 128, 256]},
        "learning_rate": {"values": [0.001, 0.000316]},
        "dropout": {"values": [0.5, 0.7]},
    },
}

# one run of the sweep: the W&B agent calls this function once per combination
def sweep_run(max_epochs=30, patience=3):
    start = time.time()
    # the agent puts the values of this combination into run.config
    run = wandb.init(project="nlp-project1-art-final")
    hidden_dim = run.config["hidden_dim"]
    learning_rate = run.config["learning_rate"]
    dropout = run.config["dropout"]
    # run name as for the other runs, and the settings that are the same in all runs
    name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}"
    run.name = name
    run.config.update({"model": "glove-gru-sweep", "rnn_type": "gru", "num_layers": 2, "inner_dim": 100, "max_epochs": max_epochs, "patience": patience, "batch_size": train_loader.batch_size, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"

    # same training as in train()
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")


In [19]:
# register the sweep at W&B and train all its combinations (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config, project="nlp-project1-art-final")
wandb.agent(sweep_id, function=sweep_run)


Create sweep with ID: c24l51cv
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/c24l51cv
glove-gru-sweep-h64-lr0.001-d0.5: best epoch 5 of 8, valid loss 0.5903, valid acc 67.80% (13.5 min)
glove-gru-sweep-h64-lr0.000316-d0.5: best epoch 7 of 10, valid loss 0.6032, valid acc 65.60% (11.8 min)
glove-gru-sweep-h128-lr0.001-d0.5: best epoch 4 of 7, valid loss 0.5872, valid acc 67.60% (14.7 min)
glove-gru-sweep-h128-lr0.000316-d0.5: best epoch 6 of 9, valid loss 0.5978, valid acc 67.90% (11.0 min)
glove-gru-sweep-h256-lr0.001-d0.5: best epoch 3 of 6, valid loss 0.6030, valid acc 65.80% (9.3 min)
glove-gru-sweep-h256-lr0.000316-d0.5: best epoch 6 of 9, valid loss 0.6052, valid acc 67.00% (15.0 min)
glove-gru-sweep-h64-lr0.001-d0.7: best epoch 6 of 9, valid loss 0.5974, valid acc 66.70% (8.1 min)
glove-gru-sweep-h64-lr0.000316-d0.7: best epoch 10 of 13, valid loss 0.6041, valid acc 66.20% (12.5 min)
glove-gru-sweep-h128-lr0.001-d0.7: b

In [20]:
# load the best checkpoint of every sweep run and sort by the validation loss, as in the section "Results"
sweep_values = sweep_config["parameters"]
results = []
for hidden_dim in sweep_values["hidden_dim"]["values"]:
    for learning_rate in sweep_values["learning_rate"]["values"]:
        for dropout in sweep_values["dropout"]["values"]:
            name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}"
            model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout)
            model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
            valid_loss, valid_acc = evaluate(model, valid_loader)
            results.append((valid_loss, valid_acc, name))

for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")


glove-gru-sweep-h128-lr0.001-d0.5: valid loss 0.5872, valid acc 67.60%
glove-gru-sweep-h64-lr0.001-d0.5: valid loss 0.5903, valid acc 67.80%
glove-gru-sweep-h128-lr0.000316-d0.7: valid loss 0.5971, valid acc 67.10%
glove-gru-sweep-h64-lr0.001-d0.7: valid loss 0.5974, valid acc 66.70%
glove-gru-sweep-h128-lr0.000316-d0.5: valid loss 0.5978, valid acc 67.90%
glove-gru-sweep-h128-lr0.001-d0.7: valid loss 0.5981, valid acc 67.20%
glove-gru-sweep-h256-lr0.000316-d0.7: valid loss 0.6012, valid acc 67.00%
glove-gru-sweep-h256-lr0.001-d0.5: valid loss 0.6030, valid acc 65.80%
glove-gru-sweep-h64-lr0.000316-d0.5: valid loss 0.6032, valid acc 65.60%
glove-gru-sweep-h256-lr0.001-d0.7: valid loss 0.6034, valid acc 65.80%
glove-gru-sweep-h64-lr0.000316-d0.7: valid loss 0.6041, valid acc 66.20%
glove-gru-sweep-h256-lr0.000316-d0.5: valid loss 0.6052, valid acc 67.00%


## Batch size
All runs above use batch size 32, as in the course exercise. Question: **does a larger batch size lower the validation loss?** The course recommends the largest batch size that fits into memory (course slides "Project Discussions" (Week 7), slides 41 and 51); this training runs on the CPU, where memory is not the limit. One sweep run tests batch size 128 with the chosen configuration, the same larger value as in the sweep of model 1 (`03_model1_embeddings.ipynb`). The learning rate stays at 0.001, so this run does not show the best result possible with this batch size. `drop_last=True` skips 33 examples per epoch (16801 = 131 x 128 + 33). The selection rule stays the same: if this run has the lowest validation loss of all candidate runs, it becomes the chosen configuration.

In [21]:
# one value: the larger batch size, with the chosen configuration
batch_size_config = {
    "name": "model2-batch-size",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"batch_size": {"values": [128]}},
}

# same training as in sweep_run(), with the training batches of this run
def batch_size_run(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=30, patience=3):
    start = time.time()
    # the agent puts the batch size of this run into run.config
    run = wandb.init(project="nlp-project1-art-final")
    batch_size = run.config["batch_size"]
    name = f"glove-gru-sweep-h{hidden_dim}-lr{learning_rate}-d{dropout}-b{batch_size}"
    run.name = name
    run.config.update({"model": "glove-gru-sweep", "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"
    # training batches with the batch size of this run
    batch_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in batch_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")

batch_size_id = wandb.sweep(batch_size_config, project="nlp-project1-art-final")
wandb.agent(batch_size_id, function=batch_size_run)


Create sweep with ID: lobgqszk
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/lobgqszk
glove-gru-sweep-h128-lr0.001-d0.5-b128: best epoch 5 of 8, valid loss 0.5969, valid acc 66.20% (5.2 min)


## Other seeds
All runs above use seed 42. Question: **how much do the validation scores of the chosen configuration change with the seed?** The number of training runs is a decision of its own, and a report can give the average of several runs to show how robust a model is (course slides "Project Discussions" (Week 7), slide 42). As for model 1 (`03_model1_embeddings.ipynb`), the chosen configuration is trained with two more seeds, 1 and 2. The seed is not tuned, because performance should not rely on it (slide 56): the chosen checkpoint stays the one with seed 42, and these runs are only reported.

In [22]:
seed_config = {
    "name": "model2-seeds",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {"seed": {"values": [1, 2]}},
}

# one run: as in sweep_run(), with the chosen configuration and the seed of this run
def seed_run(hidden_dim=128, learning_rate=0.001, dropout=0.5, max_epochs=30, patience=3):
    start = time.time()
    run = wandb.init(project="nlp-project1-art-final")
    seed = run.config["seed"]
    name = f"glove-gru-seed{seed}-h{hidden_dim}-lr{learning_rate}-d{dropout}"
    run.name = name
    run.config.update({"model": "glove-gru-seeds", "rnn_type": "gru", "hidden_dim": hidden_dim, "num_layers": 2, "inner_dim": 100, "learning_rate": learning_rate, "dropout": dropout, "max_epochs": max_epochs, "patience": patience, "batch_size": train_loader.batch_size})
    checkpoint_path = f"checkpoints/{name}.pth"

    # the seed of this run instead of 42
    torch.manual_seed(seed)
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=hidden_dim, dropout=dropout)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    minutes = (time.time() - start) / 60
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%} ({minutes:.1f} min)")

seed_id = wandb.sweep(seed_config, project="nlp-project1-art-final")
wandb.agent(seed_id, function=seed_run)


Create sweep with ID: 65qkdbkn
Sweep URL: https://forge.coreweave.com/wandb/maria-micioni-hochschule-luzern/nlp-project1-art-final/sweeps/65qkdbkn
glove-gru-seed1-h128-lr0.001-d0.5: best epoch 3 of 6, valid loss 0.6026, valid acc 66.00% (7.7 min)
glove-gru-seed2-h128-lr0.001-d0.5: best epoch 4 of 7, valid loss 0.5965, valid acc 65.20% (8.3 min)


In [23]:
# validation loss and accuracy of all three seeds, from the saved checkpoints
names = ["glove-gru-h128-lr0.001-d0.5"] + [f"glove-gru-seed{seed}-h128-lr0.001-d0.5" for seed in [1, 2]]
scores = []
for name in names:
    model = RNNClassifier(embedding_matrix, rnn_type="gru", hidden_dim=128, dropout=0.5)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    scores.append(evaluate(model, valid_loader))
losses, accuracies = zip(*scores)
# ddof=1: sample standard deviation of the three runs
print(f"chosen configuration: valid loss {np.mean(losses):.4f} (sd {np.std(losses, ddof=1):.4f}, seeds {[round(float(v), 4) for v in losses]}), valid acc {np.mean(accuracies):.2%} (sd {np.std(accuracies, ddof=1):.2%}, seeds {[round(v * 100, 2) for v in accuracies]})")


chosen configuration: valid loss 0.5954 (sd 0.0078, seeds [0.5872, 0.6026, 0.5965]), valid acc 66.27% (sd 1.22%, seeds [67.6, 66.0, 65.2])


## Model 1 vs. model 2
Both chosen models are scored on the same 1000 validation examples, so they are compared with the exact McNemar test (McNemar, 1947) on the examples that only one of them gets right.

In [24]:
from scipy.stats import binomtest

# the chosen model 1: its input (the averaged GloVe vectors) and the weights of its best epoch
X1_valid = torch.tensor(np.load("features/model1.npz")["X_valid"])
model1 = Classifier(1200, 2, inner_dim=100)
model1.load_state_dict(torch.load("checkpoints/glove-mlp-sweep-h100-lr0.000316-b32.pth", weights_only=True))
# the chosen model 2
model2 = RNNClassifier(embedding_matrix, rnn_type="gru", dropout=0.5)
model2.load_state_dict(torch.load("checkpoints/glove-gru-h128-lr0.001-d0.5.pth", weights_only=True))
# evaluation mode: no dropout
model1.eval()
model2.eval()

with torch.no_grad():
    # True where the prediction is right
    correct1 = model1(X1_valid).argmax(dim=1) == y_valid
    correct2 = model2(X_valid).argmax(dim=1) == y_valid
# & = both True, ~ = not
only_model1 = (correct1 & ~correct2).sum().item()
only_model2 = (~correct1 & correct2).sum().item()
print("both right:", (correct1 & correct2).sum().item())
print("only model 1 right:", only_model1)
print("only model 2 right:", only_model2)
print("both wrong:", (~correct1 & ~correct2).sum().item())
# exact McNemar test: binomial test on the examples that only one model gets right
print(f"p = {binomtest(only_model1, only_model1 + only_model2, 0.5).pvalue:.4f}")

both right: 506
only model 1 right: 126
only model 2 right: 170
both wrong: 198
p = 0.0123


## Check: do the observations help?
In `03_model1_embeddings.ipynb`, model 1 reached a lower validation loss with only h1 and h2 than with all four texts. Do the chosen models gain from the observations? Each validation example keeps its own h1 and h2 but gets O1 and O2 of a random validation example, 10 times (permutation feature importance, Breiman, 2001). If a model gains from the real observations, its accuracy falls. The models are not retrained: this tests the trained models, not what a model could learn from the observations.

In [25]:
# same random swaps in every run of this cell
torch.manual_seed(42)
changed1, changed2, accuracies1, accuracies2 = [], [], [], []
with torch.no_grad():
    # predictions with the real observations
    predictions1 = model1(X1_valid).argmax(dim=1)
    predictions2 = model2(X_valid).argmax(dim=1)
    for repeat in range(10):
        # for every example, the position of a random validation example
        other = torch.randperm(len(y_valid))
        # model 1: the first 600 of the 1200 numbers are O1 and O2
        swapped1 = X1_valid.clone()
        swapped1[:, :600] = X1_valid[other, :600]
        # model 2: the first 2 of the 4 texts are O1 and O2
        swapped2 = X_valid.clone()
        swapped2[:, :2] = X_valid[other, :2]
        # predictions with the observations of other examples
        new1 = model1(swapped1).argmax(dim=1)
        new2 = model2(swapped2).argmax(dim=1)
        # how many predictions change, and the accuracy
        changed1.append((new1 != predictions1).sum().item())
        changed2.append((new2 != predictions2).sum().item())
        accuracies1.append((new1 == y_valid).float().mean().item())
        accuracies2.append((new2 == y_valid).float().mean().item())
print(f"model 1: accuracy {(predictions1 == y_valid).float().mean().item():.2%} -> {np.mean(accuracies1):.2%} (lowest {min(accuracies1):.2%}, highest {max(accuracies1):.2%}), {np.mean(changed1):.0f} of 1000 predictions change")
print(f"model 2: accuracy {(predictions2 == y_valid).float().mean().item():.2%} -> {np.mean(accuracies2):.2%} (lowest {min(accuracies2):.2%}, highest {max(accuracies2):.2%}), {np.mean(changed2):.0f} of 1000 predictions change")

model 1: accuracy 63.20% -> 61.41% (lowest 60.50%, highest 62.90%), 240 of 1000 predictions change
model 2: accuracy 67.60% -> 68.10% (lowest 67.10%, highest 68.80%), 79 of 1000 predictions change


## Check: does the position of the hypotheses matter?
The classifier sees h1 and h2 in fixed input positions. If a prediction depended only on the content of the texts, swapping h1 and h2 would swap it on every example. The validation examples are scored again with h1 and h2 swapped; the models are not retrained. Scoring each hypothesis with the same classifier removes this dependence, and is tested in `04b_preprocessing_experiments.ipynb`, section "Input format variants".

In [26]:
with torch.no_grad():
    # model 1: O1, O2, then h2 before h1 (h1 = numbers 600 to 899, h2 = 900 to 1199)
    swapped1 = torch.concat([X1_valid[:, :600], X1_valid[:, 900:], X1_valid[:, 600:900]], dim=1)
    # model 2: the texts in the order O1, O2, h2, h1
    swapped2 = X_valid[:, [0, 1, 3, 2]]
    # predictions after the swap: class 0 now means h2, class 1 now means h1
    new1 = model1(swapped1).argmax(dim=1)
    new2 = model2(swapped2).argmax(dim=1)
# same class as before the swap = the prediction did not follow the hypothesis
print(f"model 1: {(new1 == predictions1).sum().item()} of 1000 predictions do not follow the swap, accuracy after the swap {(new1 == 1 - y_valid).float().mean().item():.2%}")
print(f"model 2: {(new2 == predictions2).sum().item()} of 1000 predictions do not follow the swap, accuracy after the swap {(new2 == 1 - y_valid).float().mean().item():.2%}")


model 1: 428 of 1000 predictions do not follow the swap, accuracy after the swap 61.00%
model 2: 194 of 1000 predictions do not follow the swap, accuracy after the swap 67.60%


## Conclusion
- **Chosen configuration:** GRU, RNN hidden size 128, learning rate 0.001, dropout 0.5, best epoch 4: validation loss 0.5872, accuracy 67.60%, the lowest validation loss of all 26 candidate runs. The RNN hidden size, learning rate and dropout lie inside the tested ranges. Batch size 32 is the smallest value tested; smaller values are not tested, because the course recommends the largest batch size that fits into memory.
- **In the fixed-position format, the RNN encoder beats averaging on the validation set:** all 24 RNN runs that learned have a lower validation loss than the best model 1 run (0.5872 to 0.6235 vs. 0.6445). Chosen models: 67.60% vs. 63.20% (exact McNemar test, p = 0.0123, somewhat optimistic because both were chosen on these examples). The gap remains with the same learning rate and classifier and no dropout: GRU 0.6001 and LSTM 0.6050 vs. model 1 0.6494. The experiments do not separate word order from the other differences between an RNN and averaging. With `per-choice` (`04b_preprocessing_experiments.ipynb`), averaging reaches 0.5829 and 69.10% without an RNN, so this gap may depend on the input format.
- **LSTM or GRU:** similar. The GRU has the lower validation loss in 4 of 5 settings, by at most 0.01 (course slides "Recurrent Neural Networks", slide 68).
- **Learning rate:** 0.01 is too high (both RNNs predict h1 for all examples, 50.80%); 0.0001 is worse than 0.001 (GRU 0.6193 vs. 0.6001).
- **Dropout:** 0.5 lowers the validation loss (LSTM 0.6050 to 0.5973, GRU 0.6001 to 0.5872); 0.2 does not lower it; 0.7 is worse for the GRU (0.5981) and about the same for the LSTM (0.5950).
- **Layer normalization** (classifier only): no lower validation loss (0.5931 vs. 0.5872). One learning rate, one seed.
- **Sweep:** no other combination lowers the validation loss (12 runs, 0.5872 to 0.6052), but with one seed per configuration, the others are not shown to be worse. The two repeated runs reproduce their earlier results exactly.
- **Batch size:** 128 is worse than 32 at the same learning rate (0.5969 vs. 0.5872), so 32 stays.
- **Other seeds:** over seeds 42, 1 and 2, the chosen configuration has a mean validation loss of 0.5954 (sd 0.0078) and a mean accuracy of 66.27% (sd 1.22%). Seed 42 is the best of the three, so 0.5872 and 67.60% are on the optimistic side. The seed moves the loss by up to 0.0154, about as much as the whole sweep (0.0180), so the sweep does not show a clearly best configuration. All three seeds stay ahead of all three seeds of model 1 (0.6445 to 0.6507; 62.80% to 63.50%).
- **Observations:** with O1 and O2 of random validation examples, model 1 falls from 63.20% to 61.41% and model 2 stays at 67.60% to 68.10%, so their validation accuracy may come mainly from the hypotheses. The validation set is unfiltered and the test set adversarially filtered (Bhagavatula et al., 2020; see "Data" in `README.md`), so this may not carry over.
- **Position of the hypotheses:** after swapping h1 and h2, 428 of 1000 predictions of model 1 and 194 of model 2 do not follow the swap, so both models depend on the input position, model 1 more.
- **Selection by the validation loss:** after the best epoch the loss rises while the accuracy often stays or rises, so the best epoch is selected by the loss.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier` (same as in `03_model1_embeddings.ipynb`), based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Model 2: 2-layer RNN (LSTM or GRU, PyTorch implementations) with the frozen word embeddings of model 1 and a 2-layer classifier with ReLU, as required in the Project 1 description; `nn.Embedding.from_pretrained` with `freeze=True` and `padding_idx=0` from the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html; `nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); `nn.LSTM`, `nn.GRU` and `dropout` (between the RNN layers) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html and https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html; `batch_first=True` for the input shape [batch size, sequence length, embedding dim] from the course slides "Preliminaries" (Week 1), slide 14; final hidden states of the texts as input of the classifier from the course slides "Advanced RNNs" (Week 4), slides 16-17.
- Forward pass of model 2: `pack_padded_sequence` for padding in RNNs from the course slides "Project Discussions" (Week 4), slide 17, with the lengths on the CPU and `enforce_sorted=False` from the PyTorch documentation `torch.nn.utils.rnn.pack_padded_sequence`, https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html; final hidden state of the last layer `hn[-1]` and `torch.concat(..., dim=-1)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); output `(h_n, c_n)` of the LSTM and `h_n` of the GRU from the PyTorch documentation `torch.nn.LSTM` and `torch.nn.GRU`; `isinstance` from the Python documentation, https://docs.python.org/3/library/functions.html#isinstance; counting the non-zero indices with `(text != 0).sum(dim=1)` (PyTorch documentation `torch.sum`, https://docs.pytorch.org/docs/stable/generated/torch.sum.html); test forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss, optimizer and batches: the same as in `03_model1_embeddings.ipynb`: cross-entropy loss from the course slides "Preliminaries" (Week 1), slide 27, and `nn.CrossEntropyLoss` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html; the term "logits" from the course slides "Project Discussions" (Week 5), slide 27; Adam from the course slides "Preliminaries" (Week 1), slide 29; `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; `next(iter(...))` as in the PyTorch tutorial "Datasets & DataLoaders", https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html.
- Evaluation: the same function as in `03_model1_embeddings.ipynb` (batch losses, `argmax` predictions and labels as in the validation of the course exercise "Experiment Tracking"; `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).
- Check with one batch: "overfit one batch" from Karpathy, A. (2019), *A Recipe for Training Neural Networks*, https://karpathy.github.io/2019/04/25/recipe/; training step as in the course exercise "PyTorch" (Week 1); `torch.equal` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.equal.html.
- Experiment tracking and training: the same as in `03_model1_embeddings.ipynb`: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; logging with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log; best values in the run summary as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary; training loop as in the PyTorch tutorial "Optimizing Model Parameters"; saving and loading only the parameters (`state_dict`) as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html, which is the recommended way on slide 36 of the course slides "Project Discussions" (Week 6). Early stopping on the validation loss with a patience from the course slides "Project Discussions" (Week 6), slide 38. Time per epoch with `time.time()` from the Python documentation, https://docs.python.org/3/library/time.html.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26"; LSTM vs. GRU from the course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68; the learning rate as "the most important hyperparameter" from the course slides "Project Discussions" (Week 6), slide 32.
- Dropout: dropout probability as a hyperparameter from the course slides "Preliminaries" (Week 1), slides 19-20; dropout, weight decay and normalization as regularization, and dropout not after the output, from the course slides "Project Discussions" (Week 6), slide 34; dropout from Srivastava, Hinton, Krizhevsky, Sutskever & Salakhutdinov (2014), *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*, Journal of Machine Learning Research 15, https://jmlr.org/papers/v15/srivastava14a.html; dropout only on the connections between the layers of an LSTM (not from one time step to the next) from Zaremba, Sutskever & Vinyals (2014), *Recurrent Neural Network Regularization*, https://arxiv.org/abs/1409.2329; `nn.Dropout` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html; argument `dropout` of `nn.LSTM` and `nn.GRU` ("a Dropout layer on the outputs of each LSTM layer except the last layer") from the PyTorch documentation `torch.nn.LSTM`.
- Grid search and learning rate 0.0001: grid search (all combinations) from the course slides "Project Discussions" (Week 7), slide 44; testing beyond the boundary of the tested range (slide 53), exponentially changing values (slide 52), a sensible number of epochs and sequential tuning (slide 41), and the number of runs depending on the budget (slide 55) from the same slides.
- Results and conclusion: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model" and sorting with `sorted()` as in `03_model1_embeddings.ipynb` (Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html); LSTM vs. GRU from the course slides "Recurrent Neural Networks" (Week 3), slide 68.
- Model 1 vs. model 2: exact McNemar test from McNemar, Q. (1947), *Note on the sampling error of the difference between correlated proportions or percentages*, Psychometrika 12(2), 153–157, https://doi.org/10.1007/BF02295996, as a binomial test on the examples that only one model gets right ("McNemar's test", Wikipedia, https://en.wikipedia.org/wiki/McNemar%27s_test), with `scipy.stats.binomtest` from SciPy (Virtanen et al., 2020), https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html; `&` and `~` on True/False tensors from the PyTorch documentation `torch.bitwise_and` and `torch.bitwise_not`, https://docs.pytorch.org/docs/stable/generated/torch.bitwise_and.html and https://docs.pytorch.org/docs/stable/generated/torch.bitwise_not.html; loading the checkpoints as in the section "Results"; `model.eval()` and `torch.no_grad()` as in `evaluate()`.
- Check of the observations: permutation feature importance from Breiman, L. (2001), *Random Forests*, Machine Learning 45(1), 5–32, https://doi.org/10.1023/A:1010933404324, and the scikit-learn documentation "Permutation feature importance", https://scikit-learn.org/stable/modules/permutation_importance.html; `torch.randperm`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.randperm.html; `Tensor.clone`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.clone.html; `min()` and `max()`, Python documentation, https://docs.python.org/3/library/functions.html; origin of the splits: Bhagavatula et al. (2020), Table 6, and course slides "Course Project HS26", slide 8.
- Layer normalization: `nn.LayerNorm`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html, `nn.Sequential`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html, and `nn.GRU` (no option for layer normalization), https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html, from the PyTorch documentation; before the nonlinearity and the layer normalized RNN from Ba, Kiros & Hinton (2016), *Layer Normalization*, https://arxiv.org/abs/1607.06450; normalization as an option and layer norm (yes/no) as an experiment from the course slides "Project Discussions" (Week 6), slide 34, and (Week 7), slide 44; subclass from the Python tutorial "Classes", https://docs.python.org/3/tutorial/classes.html#inheritance. Check: `torch.numel` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.numel.html; `weight_ih_l0`, the variable `weight_ih_l[k]` in the PyTorch documentation `torch.nn.GRU`, https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options (`method`, `metric`, `parameters`, `values`), https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough (`wandb.sweep`, `wandb.agent` with `function`, `wandb.init` inside the function, `run.config`), https://docs.coreweave.com/models/sweeps/walkthrough; `run.config.update`, W&B documentation on the run config, https://docs.coreweave.com/models/track/config; `run.name`, W&B documentation `Run`, https://docs.coreweave.com/models/ref/python/experiments/run; grid search (slide 44), sequential tuning (slide 41), finer steps such as 1e-3.5 (slide 52) and the number of trials (slide 55) from the same slides.
- Batch size: the largest batch size that fits into memory from the course slides "Project Discussions" (Week 7), slides 41 and 51; training batches with `DataLoader` as in the section "Loss, optimizer and batches".
- Other seeds: reporting the average of several runs and the debate about tuning the seed from the course slides "Project Discussions" (Week 7), slides 42 and 56; `np.std` with `ddof`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.std.html; `zip(*scores)`, Python documentation, https://docs.python.org/3/library/functions.html#zip.
- Check of the position: `torch.concat` as in the section "Model 2: RNN + classifier"; indexing with a list of positions, NumPy documentation "Indexing on ndarrays", https://numpy.org/doc/stable/user/basics.indexing.html, which works the same on PyTorch tensors.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.